In [ ]:
#nspect all 3 datasets
import pandas as pd

# 1) Original crop dataset
df1 = pd.read_excel("Crop_recommendation.xlsx")
print("== Crop_recommendation.xlsx ==")
print("Shape:", df1.shape)
print("Columns:", df1.columns.tolist())
print(df1.head(3), "\n")

# 2) crop_yield.csv
df2 = pd.read_csv("crop_yield.csv")
print("== crop_yield.csv ==")
print("Shape:", df2.shape)
print("Columns:", df2.columns.tolist())
print(df2.head(3), "\n")

# 3) sensor_Crop_Dataset
df3 = pd.read_csv("sensor_Crop_Dataset (1).csv")
print("== sensor_Crop_Dataset (1).csv ==")
print("Shape:", df3.shape)
print("Columns:", df3.columns.tolist())
print(df3.head(3))


== Crop_recommendation.xlsx ==
Shape: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice 

== crop_yield.csv ==
Shape: (1000000, 10)
Columns: ['Region', 'Soil_Type', 'Crop', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition', 'Days_to_Harvest', 'Yield_tons_per_hectare']
  Region Soil_Type    Crop  Rainfall_mm  Temperature_Celsius  Fertilizer_Used  \
0   West     Sandy  Cotton   897.077239            27.676966            False   
1  South      Clay    Rice   992.673282            18.026142             True   
2  North      Loam  Barley   147.998025            29.794042            False   

   Irrigation_Used Weather_Condition  Days_to_Harvest  Yi

In [3]:
#imports & eval helper
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# TabNet (optional)
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    TABNET_AVAILABLE = True
except Exception:
    TABNET_AVAILABLE = False

def evaluate_dl(model, X_test, y_test, label_encoder):
    y_true = np.argmax(y_test, axis=1)
    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
    return acc


In [8]:
#train_all_models_on_dataset
def train_all_models_on_dataset(file_path, target_col):
    print(f"\n========== Training on dataset: {file_path} ==========")

    # Load file
    if file_path.endswith(".xlsx"):
        df = pd.read_excel(file_path)
    elif file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        raise ValueError("Use .csv or .xlsx files")

    # Check target column exists
    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found in {file_path}. "
            f"Columns: {df.columns.tolist()}"
        )

    # --------------------------
    # 1. SPLIT X AND Y
    # --------------------------
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Encode labels
    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    y_ohe = tf.keras.utils.to_categorical(y_enc)

    # --------------------------
    # 2. One-Hot Encode Categorical Feature Columns
    # --------------------------
    cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

    if len(cat_cols) > 0:
        print("Encoding categorical feature columns:", cat_cols)
        X = pd.get_dummies(X, columns=cat_cols)

    # --------------------------
    # 3. SCALE FEATURES
    # --------------------------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # --------------------------
    # 4. TRAIN-TEST SPLIT
    # --------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_ohe, test_size=0.2, random_state=42, stratify=y_enc
    )

    input_dim = X_train.shape[1]
    num_classes = y_train.shape[1]

    print(f"Samples: {df.shape[0]}, Features: {input_dim}, Classes: {num_classes}")

    results = {}

    # ====================================================
    # 1. DNN / MLP
    # ====================================================
    print("\n[1] DNN / MLP")
    mlp = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    mlp.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    mlp.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    print("DNN / MLP results:")
    acc_mlp = evaluate_dl(mlp, X_test, y_test, le)
    results["DNN / MLP"] = acc_mlp

    # ====================================================
    # 2. Autoencoder + DNN
    # ====================================================
    print("\n[2] Autoencoder + DNN")

    encoding_dim = 4
    inp = layers.Input(shape=(input_dim,))
    enc = layers.Dense(32, activation='relu')(inp)
    enc = layers.Dense(encoding_dim, activation='relu')(enc)
    dec = layers.Dense(32, activation='relu')(enc)
    dec = layers.Dense(input_dim, activation='linear')(dec)

    autoencoder = models.Model(inp, dec)
    encoder = models.Model(inp, enc)
    autoencoder.compile(optimizer='adam', loss='mse')

    ae_es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    autoencoder.fit(
        X_train, X_train,
        validation_split=0.1,
        epochs=50,
        batch_size=32,
        callbacks=[ae_es],
        verbose=0
    )

    X_train_enc = encoder.predict(X_train)
    X_test_enc = encoder.predict(X_test)

    clf = models.Sequential([
        layers.Dense(32, activation='relu', input_shape=(encoding_dim,)),
        layers.Dense(num_classes, activation='softmax')
    ])
    clf.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    clf_es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    clf.fit(
        X_train_enc, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=32,
        callbacks=[clf_es],
        verbose=0
    )

    print("Autoencoder + DNN results:")
    acc_auto = evaluate_dl(clf, X_test_enc, y_test, le)
    results["Autoencoder + DNN"] = acc_auto

    # ====================================================
    # 3. Keras Transformer
    # ====================================================
    print("\n[3] Keras Transformer")

    X_train_tr = X_train.reshape(-1, input_dim, 1)
    X_test_tr = X_test.reshape(-1, input_dim, 1)

    d_model = 32
    num_heads = 4

    tinp = layers.Input(shape=(input_dim, 1))
    x = layers.Dense(d_model)(tinp)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(x, x)
    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    ffn = layers.Dense(64, activation='relu')(x)
    ffn = layers.Dense(d_model, activation='relu')(ffn)
    x = layers.Add()([x, ffn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    transformer = models.Model(tinp, out)
    transformer.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    tr_es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    transformer.fit(
        X_train_tr, y_train,
        validation_split=0.1,
        epochs=50,
        batch_size=32,
        callbacks=[tr_es],
        verbose=0
    )

    print("Keras Transformer results:")
    acc_trans = evaluate_dl(transformer, X_test_tr, y_test, le)
    results["Keras Transformer"] = acc_trans

    # ====================================================
    # 4. TabNet (if available)
    # ====================================================
    if TABNET_AVAILABLE:
        try:
            print("\n[4] TabNet")

            X_raw = X.values
            y_raw = y_enc

            X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
                X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
            )

            tabnet = TabNetClassifier(verbose=0)
            tabnet.fit(
                X_train_t, y_train_t,
                eval_set=[(X_test_t, y_test_t)],
                patience=10,
                max_epochs=100
            )

            y_pred_t = tabnet.predict(X_test_t)
            acc_tabnet = accuracy_score(y_test_t, y_pred_t)

            print("TabNet Accuracy:", acc_tabnet)
            print(classification_report(y_test_t, y_pred_t, target_names=le.classes_))

            results["TabNet"] = acc_tabnet

        except Exception as e:
            print("TabNet failed:", e)

    # --------------------------
    # SUMMARY TABLE
    # --------------------------
    print("\n=== Summary for", file_path, "===")
    print(pd.DataFrame(results, index=["Accuracy"]).T.sort_values(by="Accuracy", ascending=False))

    return results


In [9]:
#fill your target columns and run
DATASETS = [
    ("Crop_recommendation.xlsx", "label"),          # we know this one
    ("crop_yield.csv", "Crop"), # <-- CHANGE THIS
    ("sensor_Crop_Dataset (1).csv", "Crop")  # <-- CHANGE THIS
]

all_results = {}

for file_path, target_col in DATASETS:
    res = train_all_models_on_dataset(file_path, target_col)
    all_results[file_path] = res

# Overall comparison
rows = []
for file_path, res in all_results.items():
    for model_name, acc in res.items():
        rows.append({"dataset": file_path, "model": model_name, "accuracy": acc})

overall_df = pd.DataFrame(rows)
overall_df_pivot = overall_df.pivot(index="model", columns="dataset", values="accuracy")
overall_df_pivot



========== Training on dataset: Crop_recommendation.xlsx ==========
Samples: 2200, Features: 7, Classes: 22

[1] DNN / MLP


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


DNN / MLP results:
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Accuracy: 0.9863636363636363
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      1.00      1.00        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       0.91      1.00      0.95        20
      grapes       1.00      1.00      1.00        20
        jute       0.90      0.95      0.93        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      1.00      1.00        20
       maize       1.00      0.90      0.95        20
       mango       1.00      1.00      1.00        20
   mothbeans       1.00      1.00      1.00        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       0.95      1.00      0.98    

c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Autoencoder + DNN results:
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Accuracy: 0.8568181818181818
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      0.95      0.97        20
   blackgram       0.70      0.95      0.81        20
    chickpea       0.95      1.00      0.98        20
     coconut       0.95      1.00      0.98        20
      coffee       0.79      0.75      0.77        20
      cotton       0.75      0.75      0.75        20
      grapes       1.00      1.00      1.00        20
        jute       0.78      0.90      0.84        20
 kidneybeans       1.00      0.95      0.97        20
      lentil       0.93      0.65      0.76        20
       maize       0.94      0.85      0.89        20
       mango       0.71      0.75      0.73        20
   mothbeans       0.69      0.45      0.55        20
    mungbean       0.95      1.00      0.98        20
   muskmelon       0.95      1.00      

...

Autoencoder + DNN results:
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 600us/step
Accuracy: 1.0
              precision    recall  f1-score   support

       Maize       1.00      1.00      1.00       670
      Potato       1.00      1.00      1.00       672
        Rice       1.00      1.00      1.00       654
   Sugarcane       1.00      1.00      1.00       657
      Tomato       1.00      1.00      1.00       669
       Wheat       1.00      1.00      1.00       678

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000


[3] Keras Transformer
Keras Transformer results:
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.1695
              precision    recall  f1-score   support

       Maize       0.00      0.00      0.00       670
      Potato       0.00      0.00      0.00       672
        Rice       0.00      0.00      0.00       654
   Sugarcane       0.00      0.00      0.00     

c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is


Early stopping occurred at epoch 27 with best_epoch = 17 and best_val_0_accuracy = 1.0
TabNet failed: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found object

=== Summary for sensor_Crop_Dataset (1).csv ===
                   Accuracy
DNN / MLP            1.0000
Autoencoder + DNN    1.0000
Keras Transformer    0.1695


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


dataset,Crop_recommendation.xlsx,crop_yield.csv,sensor_Crop_Dataset (1).csv
model,,,
Autoencoder + DNN,0.856818,0.166775,1.0000
DNN / MLP,0.986364,0.166790,1.0000
Keras Transformer,0.650000,0.166790,0.1695
TabNet,0.102273,NaN,NaN


In [10]:
#which column are non numeric
import pandas as pd

df2 = pd.read_csv("crop_yield.csv")
print("crop_yield dtypes:\n", df2.dtypes, "\n")

df3 = pd.read_csv("sensor_Crop_Dataset (1).csv")
print("sensor_Crop dtypes:\n", df3.dtypes)


crop_yield dtypes:
 Region                     object
Soil_Type                  object
Crop                       object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64
dtype: object 

sensor_Crop dtypes:
 Nitrogen       float64
Phosphorus     float64
Potassium      float64
Temperature    float64
Humidity       float64
pH_Value       float64
Rainfall       float64
Crop            object
Soil_Type       object
Variety         object
dtype: object
